# Regime Paper Cases

This notebook defines the five case studies that should anchor the paper section on regime-aware forecasting.

Design principle:
- dense / stable demand -> `SARIMAX`
- transition / mixed demand -> `Hurdle`
- intermittent / low-rotation demand -> `TSB`


In [7]:
import sys
print(sys.executable)
!"{sys.executable}" -m pip install pandas



c:\Users\braya\AppData\Local\Programs\Python\Python311\python.exe



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
!"{sys.executable}" -m pip install -r requirements.txt


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from pathlib import Path
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)



Repo root: C:\Users\braya\Documents\Research\aurex-demand-forecasting-main


In [10]:
paper_cases = pd.DataFrame([
    {
        'dataset': 'FAVORITA',
        'series_id': 'item_268443',
        'expected_regime': 'stable',
        'demand_role': 'high-demand reference',
        'paper_model': 'SARIMAX',
        'why_model': 'Dense and comparatively stable daily demand should favor a continuous time-series model.',
        'source_plot': 'reports/regime_eda_final/plots/FAVORITA__item_268443.png',
    },
    {
        'dataset': 'FAVORITA',
        'series_id': 'item_789224',
        'expected_regime': 'transition',
        'demand_role': 'medium-demand transition case',
        'paper_model': 'Hurdle',
        'why_model': 'Transition behavior can keep non-zero activity while becoming irregular; Hurdle handles changing occurrence more robustly than plain SARIMAX.',
        'source_plot': 'reports/regime_eda_final/plots/FAVORITA__item_789224.png',
    },
    {
        'dataset': 'M5_WALMART',
        'series_id': 'HOUSEHOLD_1_187_WI_1_validation',
        'expected_regime': 'transition',
        'demand_role': 'retail transition reference',
        'paper_model': 'Hurdle',
        'why_model': 'The repo already highlights this item as a transition example; it is a good regime-selection demonstration case.',
        'source_plot': 'reports/regime_eda_final/plots/M5_WALMART__HOUSEHOLD_1_187_WI_1_validation.png',
    },
    {
        'dataset': 'AMAZON_2023',
        'series_id': 'B00QWO9P0O',
        'expected_regime': 'intermittent',
        'demand_role': 'low-demand intermittent case',
        'paper_model': 'TSB',
        'why_model': 'High sparsity and low-rotation demand make this a clean intermittent-demand example for TSB.',
        'source_plot': 'reports/regime_eda_final/plots/AMAZON_2023__B00QWO9P0O.png',
    },
    {
        'dataset': 'AMAZON_2023',
        'series_id': 'B0BZTLKX7H',
        'expected_regime': 'intermittent',
        'demand_role': 'very-low-demand stress test',
        'paper_model': 'TSB',
        'why_model': 'This item is already used in the one-year comparison notebook as an intermittent stress test proxy.',
        'source_plot': 'reports/regime_eda_final/plots/AMAZON_2023__B0BZTLKX7H.png',
    },
])

paper_cases


,dataset,series_id,expected_regime,demand_role,paper_model,why_model,source_plot
0,FAVORITA,item_268443,stable,high-demand reference,SARIMAX,Dense and comparatively stable daily demand sh...,reports/regime_eda_final/plots/FAVORITA__item_...
1,FAVORITA,item_789224,transition,medium-demand transition case,Hurdle,Transition behavior can keep non-zero activity...,reports/regime_eda_final/plots/FAVORITA__item_...
2,M5_WALMART,HOUSEHOLD_1_187_WI_1_validation,transition,retail transition reference,Hurdle,The repo already highlights this item as a tra...,reports/regime_eda_final/plots/M5_WALMART__HOU...
3,AMAZON_2023,B00QWO9P0O,intermittent,low-demand intermittent case,TSB,High sparsity and low-rotation demand make thi...,reports/regime_eda_final/plots/AMAZON_2023__B0...
4,AMAZON_2023,B0BZTLKX7H,intermittent,very-low-demand stress test,TSB,This item is already used in the one-year comp...,reports/regime_eda_final/plots/AMAZON_2023__B0...


## Link to Current Repo Logic

The current selector in `src/models/regime_forecast_engine.py` only switches between `SARIMAX` and `Hurdle`.

For the paper, the clean interpretation is:
- keep the selector story for dense vs zero-heavy series
- add `TSB` as a focused experimental model for the clearly intermittent cases


In [11]:
selector_rules = pd.DataFrame([
    {
        'rule': 'zero_rate >= 0.20',
        'current_engine_choice': 'Hurdle',
        'paper_interpretation': 'zero-heavy demand; Hurdle baseline and TSB comparison are justified',
    },
    {
        'rule': 'zero_rate >= 0.10 and CV high',
        'current_engine_choice': 'Hurdle',
        'paper_interpretation': 'mixed/intermittent dynamics; compare Hurdle vs TSB if sparse enough',
    },
    {
        'rule': 'otherwise',
        'current_engine_choice': 'SARIMAX or rolling-validation winner',
        'paper_interpretation': 'dense demand; SARIMAX is the natural showcase model',
    },
])

selector_rules


,rule,current_engine_choice,paper_interpretation
0,zero_rate >= 0.20,Hurdle,zero-heavy demand; Hurdle baseline and TSB com...
1,zero_rate >= 0.10 and CV high,Hurdle,mixed/intermittent dynamics; compare Hurdle vs...
2,otherwise,SARIMAX or rolling-validation winner,dense demand; SARIMAX is the natural showcase ...
